# Lab 01 · Importar y unir

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 1*

Trabaja sobre tu propia copia del notebook. Todo lo que escribas queda en ella.

In [ ]:
#@title De qué se trata este lab { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">De qué se trata este lab</div><strong>Preguntas que vamos a responder</strong>
<ul>
<li>Con qué tecnologías está hecho el parque de generación del país</li>
<li>Cómo se trae a Python un dato que está en Excel, en una base de datos o en un servicio web</li>
<li>Qué pasa cuando dos tablas no calzan del todo</li>
</ul>
<strong>Al terminar vas a poder</strong>
<ul>
<li>Cargar tablas desde CSV, Excel, JSON, una API y una base de datos SQL</li>
<li>Filtrar, agrupar y resumir con pandas</li>
<li>Unir dos tablas eligiendo el tipo de join que corresponde</li>
<li>Reconocer cuándo conviene Polars en vez de pandas</li>
</ul></div>"""))

In [ ]:
#@title Datos del curso { display-mode: "form" }
#@markdown Corre esta celda. Deja listos los archivos del Observatorio de Datos Energeticos.
import numpy as np, pandas as pd, os, json, sqlite3
if not os.path.exists("centrales.csv"):
    rng = np.random.default_rng(2026)
    centrales = pd.DataFrame([
     ("Central Rio Manso Alto","hidro","Biobio",420,2004),("Central Salto Verde","hidro","Los Lagos",310,1998),
     ("Central Aguas Claras","hidro","Biobio",180,2011),("Central Vega Azul","hidro","Los Lagos",95,2016),
     ("Central Tres Saltos","hidro","Biobio",260,1995),
     ("Parque Solar Pampa Alta","solar","Antofagasta",230,2019),("Parque Solar Llano Seco","solar","Atacama",180,2020),
     ("Parque Solar Sol Naciente","solar","Antofagasta",145,2021),("Parque Solar Quebrada Honda","solar","Atacama",95,2022),
     ("Parque Solar Altiplano","solar","Antofagasta",310,2023),
     ("Eolica Cerro Negro","eolica","Coquimbo",160,2017),("Eolica Punta Ventosa","eolica","Coquimbo",120,2018),
     ("Eolica Loma Fria","eolica","Valparaiso",85,2020),("Eolica Campo Abierto","eolica","Coquimbo",200,2021),
     ("Termoelectrica Bahia Norte","gas","Valparaiso",375,2008),("Termoelectrica Puerto Sur","gas","Biobio",290,2012),
     ("Termoelectrica Valle Central","gas","Metropolitana",210,2006),
     ("Carboelectrica Costa Brava","carbon","Biobio",480,2001),("Carboelectrica Roca Gris","carbon","Antofagasta",350,1999),
     ("Diesel Respaldo Cordillera","diesel","Metropolitana",45,2014),
    ], columns=["central","tecnologia","region","potencia_mw","anio_inicio"])
    fechas = pd.date_range("2024-01-01","2024-12-31",freq="D")
    perfil = np.array([0,0,0,0,0,0,.05,.18,.38,.58,.75,.87,.93,.9,.8,.63,.42,.2,.05,0,0,0,0,0])
    filas=[]
    for _,c in centrales.iterrows():
        p,t = c.potencia_mw, c.tecnologia
        for f in fechas:
            est = 1+0.25*np.cos(2*np.pi*(f.dayofyear-15)/365)
            if t=="solar": base = p*perfil*0.30*est*rng.uniform(.8,1.1)
            elif t=="eolica": base = p*0.36*rng.uniform(.15,1.6,24)
            elif t=="hidro": base = p*0.55*(2-est)*rng.uniform(.9,1.1,24)
            elif t=="gas": base = p*0.68*rng.uniform(.9,1.05,24)
            elif t=="carbon": base = p*0.65*rng.uniform(.95,1.02,24)
            else:
                base = np.zeros(24); base[18:23] = p*0.55*rng.uniform(.8,1,5)
            filas.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                       "central":c.central,"mwh":np.clip(base,0,p).round(2)}))
    centrales.to_csv("centrales.csv", index=False)
    pd.concat(filas, ignore_index=True).to_csv("generacion.csv", index=False)

    # Excel con dos hojas, la segunda con notas en texto libre
    with pd.ExcelWriter("centrales.xlsx") as w:
        centrales.to_excel(w, sheet_name="centrales", index=False)
        pd.DataFrame({"nota":["Potencias declaradas al 31 de diciembre de 2024",
                              "Las centrales de pasada se informan con su potencia maxima"]}
                     ).to_excel(w, sheet_name="notas", index=False)

    # Demanda por región, base de datos SQLite
    regs = ["Antofagasta","Atacama","Coquimbo","Valparaiso","Metropolitana","Biobio","Los Lagos"]
    pobl = [700000,320000,850000,1900000,8100000,1700000,900000]
    perfil_d = np.array([.72,.68,.66,.65,.66,.70,.78,.88,.95,.98,1.0,1.02,1.03,1.0,.97,.96,.97,1.0,1.06,1.10,1.08,.98,.88,.79])
    dem=[]
    for r,p in zip(regs,pobl):
        base_r = p/8000
        for f in fechas:
            inv = 1+0.18*np.cos(2*np.pi*(f.dayofyear-190)/365)
            finde = 0.92 if f.dayofweek>=5 else 1.0
            v = base_r*perfil_d*inv*finde*rng.uniform(.97,1.03,24)
            dem.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                     "region":r,"mwh":v.round(2)}))
    demanda = pd.concat(dem, ignore_index=True)
    demanda.to_csv("demanda.csv", index=False)
    con = sqlite3.connect("demanda.db")
    demanda.to_sql("demanda", con, index=False, if_exists="replace")
    pd.DataFrame({"region":regs,"poblacion":pobl}).to_sql("regiones", con, index=False, if_exists="replace")
    con.close()

    # Precios de nudo de enero, como los entregaría una API REST
    ene = demanda[demanda["fecha"].str.startswith("2024-01")]
    pr = ene.assign(precio_usd_mwh=(40 + ene["mwh"]/ene["mwh"].max()*110
                                    + rng.normal(0,6,len(ene))).clip(40,180).round(2))
    with open("precios_nudo.json","w") as f:
        json.dump({"metadata":{"fuente":"Observatorio de Datos Energeticos",
                               "fecha_consulta":"2024-02-01","unidad":"USD por MWh"},
                   "datos": pr[["fecha","hora","region","precio_usd_mwh"]].to_dict("records")},
                  f)
print("Datos listos")


## 1. Lo primero, una tabla

In [ ]:
import pandas as pd

# read_csv lee un archivo separado por comas y lo convierte en una tabla.
# Esa tabla se llama DataFrame y es con lo que vas a trabajar todo el curso.
centrales = pd.read_csv("centrales.csv")

centrales

In [ ]:
# Tres preguntas que uno le hace a cualquier tabla apenas la abre.
print("Filas y columnas", centrales.shape)
print("Columnas        ", list(centrales.columns))
print("Potencia total  ", centrales["potencia_mw"].sum(), "MW")

In [ ]:
# Una columna se pide con su nombre entre corchetes.
# Un filtro es una condición dentro de los corchetes, que deja pasar solo las filas que la cumplen.
centrales[centrales["tecnologia"] == "solar"]

In [ ]:
# groupby junta las filas que comparten un valor y después las resume.
por_tecnologia = centrales.groupby("tecnologia")["potencia_mw"].sum().sort_values()

print(por_tecnologia)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(por_tecnologia.index, por_tecnologia.values, color="#4C78A8")
ax.set_title("La hidro encabeza y la solar ya pasó al carbón")
ax.set_xlabel("Potencia instalada (MW)")
plt.show()

In [ ]:
#@title Ojo con esta cifra { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Ojo con esta cifra</div><p>Este parque de veinte centrales es inventado para el curso y no reproduce la matriz chilena. Acá la hidro encabeza y la solar queda cuarta.</p><p>En el sistema real de 2025 la primera fue la solar fotovoltaica, con 20.649 GWh de 85 TWh generados, seguida del carbón con 15.368 GWh. Lo que sí funciona igual que en la realidad es la relación entre potencia instalada y energía generada, que es lo que este bloque quiere mostrar.</p></div>"""))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Cambia la columna del resumen</strong>
<p>La celda de abajo es la misma de arriba, agrupando por región. Cámbiala por <code>anio_inicio</code> y vuelve a correrla para ver en qué años se construyó el parque.</p></div>"""))

In [ ]:
columna = "region"   # cambiala por tecnologia o por anio_inicio

resumen = centrales.groupby(columna)["potencia_mw"].sum().sort_values()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(resumen.index.astype(str), resumen.values, color="#4C78A8")
ax.set_title(f"Potencia instalada por {columna}")
ax.set_xlabel("Potencia instalada (MW)")
plt.show()

## 2. El CSV y sus tres trampas

In [ ]:
# La generación de cada central en cada hora del año.
generacion = pd.read_csv("generacion.csv")

print("Filas y columnas", generacion.shape)
generacion.head()

In [ ]:
# Fíjate en el tipo de cada columna. La fecha llegó como texto, no como fecha.
print(generacion.dtypes)

In [ ]:
# parse_dates le dice a pandas que esa columna son fechas de verdad.
generacion = pd.read_csv("generacion.csv", parse_dates=["fecha"])

print(generacion.dtypes)
print()
print("Ahora se le pueden pedir cosas a la fecha")
print("Primer día  ", generacion["fecha"].min().date())
print("Último día  ", generacion["fecha"].max().date())

## 3. Excel

In [ ]:
# read_excel funciona igual que read_csv. La diferencia es que un Excel puede traer varias hojas.
fichas = pd.read_excel("centrales.xlsx")

fichas.head(3)

In [ ]:
# sheet_name elige la hoja. Sin ese argumento pandas trae la primera.
notas = pd.read_excel("centrales.xlsx", sheet_name="notas")

notas

In [ ]:
# Con sheet_name=None trae todas las hojas de una vez, en un diccionario.
todas = pd.read_excel("centrales.xlsx", sheet_name=None)

print("Hojas del archivo", list(todas.keys()))

## 4. Una API REST

In [ ]:
import json

# Contra una API real esta línea sería
#   respuesta = requests.get("https://api.ejemplo.cl/precios").json()
with open("precios_nudo.json") as f:
    respuesta = json.load(f)

# Un JSON casi nunca es una tabla. Primero hay que mirar cómo viene armado.
print("Llaves de la respuesta", list(respuesta.keys()))
print(respuesta["metadata"])

In [ ]:
# json_normalize toma la lista de registros y la aplana a tabla.
precios = pd.json_normalize(respuesta["datos"])

print("Filas y columnas", precios.shape)
precios.head()

## 5. Una base de datos

In [ ]:
import sqlite3

# SQLite guarda toda la base en un archivo. Postgres u Oracle se conectan por red,
# pero la línea de pandas que viene después es exactamente la misma.
conexion = sqlite3.connect("demanda.db")

pd.read_sql("SELECT * FROM regiones", conexion)

In [ ]:
# El filtro se escribe en SQL y lo resuelve la base, así viaja mucho menos dato.
consulta = """
    SELECT region, SUM(mwh) AS demanda_total
    FROM demanda
    GROUP BY region
    ORDER BY demanda_total DESC
"""
pd.read_sql(consulta, conexion)

## 6. Unir dos tablas

In [ ]:
# Para que se note la diferencia, trabajamos con un día y con una ficha incompleta.
un_dia = generacion[generacion["fecha"] == "2024-06-15"]

# Sacamos tres centrales de la lista de fichas y agregamos una que no genera.
fichas_parciales = centrales[~centrales["central"].isin(
    ["Eolica Loma Fria", "Central Vega Azul", "Diesel Respaldo Cordillera"])]
fichas_parciales = pd.concat([fichas_parciales, pd.DataFrame(
    [{"central":"Central Proyecto Futuro","tecnologia":"hidro","region":"Biobio",
      "potencia_mw":150,"anio_inicio":2026}])], ignore_index=True)

print("Generación de ese día", un_dia.shape[0], "filas,", un_dia["central"].nunique(), "centrales")
print("Fichas disponibles   ", fichas_parciales.shape[0], "centrales")

In [ ]:
# inner se queda solo con las centrales que están en las dos tablas.
inner = pd.merge(un_dia, fichas_parciales, on="central", how="inner")

print("inner", inner.shape[0], "filas de", un_dia.shape[0])
print("Perdió en silencio", un_dia.shape[0] - inner.shape[0], "filas de generación")

In [ ]:
# left conserva toda la generación y deja vacía la ficha cuando no existe.
left = pd.merge(un_dia, fichas_parciales, on="central", how="left")

print("left", left.shape[0], "filas")
print("Filas sin ficha", left["tecnologia"].isna().sum())
left[left["tecnologia"].isna()]["central"].unique()

In [ ]:
# outer conserva todo de ambos lados. Con indicator queda anotado de dónde salió cada fila.
outer = pd.merge(un_dia, fichas_parciales, on="central", how="outer", indicator=True)

print(outer["_merge"].value_counts())

In [ ]:
# cross cruza todo con todo. Sirve para armar la grilla completa y ver qué falta.
horas = pd.DataFrame({"hora_del_dia": range(24)})
grilla = pd.merge(fichas_parciales[["central"]], horas, how="cross")

print("Grilla completa", grilla.shape[0], "filas =", fichas_parciales.shape[0], "centrales por 24 horas")

In [ ]:
# Así se revisa antes de unir.
print("Fichas repetidas", fichas_parciales["central"].duplicated().sum())

# Y así se ve el daño cuando las hay.
con_repetida = pd.concat([fichas_parciales, fichas_parciales.head(1)], ignore_index=True)
inflado = pd.merge(un_dia, con_repetida, on="central", how="inner")
print("Join normal ", inner.shape[0], "filas")
print("Join con una ficha repetida", inflado.shape[0], "filas")

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Elige el join que corresponde</strong>
<p>Queremos la generación total por tecnología de ese día, sin perder ninguna fila de generación. Cambia el <code>how</code> de la celda de abajo hasta que el total coincida con la generación real del día, que está impresa arriba del resultado.</p></div>"""))

In [ ]:
how = "inner"   # pruebalo con inner, left y outer

unido = pd.merge(un_dia, fichas_parciales, on="central", how=how)

print("Generación real de ese día", round(un_dia["mwh"].sum(), 1), "MWh")
print("Generación según el join  ", round(unido["mwh"].sum(), 1), "MWh")
print()
print(unido.groupby("tecnologia")["mwh"].sum().round(1).sort_values(ascending=False))

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Limpia la tabla antes de unir</strong>
<p>La tabla <code>fichas_sucias</code> trae centrales repetidas, como llegan en la vida real. Completa la linea marcada con <code>drop_duplicates</code> para que el join deje el mismo numero de filas que la generacion del dia.</p></div>"""))

In [ ]:
# Una tabla de fichas como llega en la práctica, con dos centrales repetidas.
fichas_sucias = pd.concat([fichas_parciales, fichas_parciales.head(2)], ignore_index=True)
print("Fichas repetidas en la tabla sucia", fichas_sucias["central"].duplicated().sum())

# Cambia esta línea por fichas_sucias.drop_duplicates(subset="central")
fichas_limpias = fichas_sucias

unido = pd.merge(un_dia, fichas_limpias, on="central", how="left")
print("Filas de generación del día", un_dia.shape[0])
print("Filas después del join    ", unido.shape[0])
print("Generación real ", round(un_dia["mwh"].sum(), 1), "MWh")
print("Generación unida", round(unido["mwh"].sum(), 1), "MWh")

## 7. Cuando la tabla no cabe

In [ ]:
# Un archivo grande de verdad, para medir. Son cinco años de 100 centrales.
import numpy as np, os
if not os.path.exists("generacion_grande.csv"):
    rng = np.random.default_rng(7)
    n = 100 * 5 * 365 * 24
    pd.DataFrame({
        "fecha": np.repeat(pd.date_range("2020-01-01", periods=5*365).astype(str), 100*24),
        "hora": np.tile(np.repeat(np.arange(24), 100), 5*365),
        "central": np.tile([f"Central {i:03d}" for i in range(100)], 5*365*24),
        "mwh": rng.uniform(0, 400, n).round(2),
    }).to_csv("generacion_grande.csv", index=False)

print("Tamaño del archivo", round(os.path.getsize("generacion_grande.csv")/1e6, 1), "MB")

In [ ]:
%%time
# pandas leyendo el archivo grande
grande_pd = pd.read_csv("generacion_grande.csv")
print(grande_pd.shape)

In [ ]:
# Se importa aparte, para que el reloj de la celda siguiente mida solo la lectura.
import polars as pl
print("Versión de Polars", pl.__version__)

In [ ]:
%%time
# Polars leyendo exactamente el mismo archivo
grande_pl = pl.read_csv("generacion_grande.csv")
print(grande_pl.shape)

In [ ]:
%%time
# El mismo resumen en pandas
print(grande_pd.groupby("central")["mwh"].sum().head(3))

In [ ]:
%%time
# El mismo resumen en Polars. Cambia cómo se escribe, no lo que hace.
print(grande_pl.group_by("central").agg(pl.col("mwh").sum()).head(3))

In [ ]:
#@title Puntos clave { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Puntos clave</div><ul>
<li><code>read_csv</code>, <code>read_excel</code>, <code>json_normalize</code> y <code>read_sql</code> traen datos de cualquier origen a la misma estructura, el DataFrame</li>
<li>Las fechas hay que pedirlas con <code>parse_dates</code>, porque llegan como texto</li>
<li>El filtro y el resumen se pueden hacer en la base con SQL o en Python con pandas, y conviene hacerlo donde estén los datos</li>
<li><code>inner</code> pierde filas, <code>left</code> las conserva, <code>outer</code> conserva todo y <code>cross</code> cruza todo con todo</li>
<li>Una clave repetida infla el join sin avisar, así que se revisa antes</li>
<li>Polars es para cuando el archivo pesa, no para reemplazar a pandas</li>
</ul>
<p>En el Lab 02 vamos a trabajar esa tabla unida, filtrando, agrupando y creando columnas nuevas.</p></div>"""))